# Notebook 2 — Veri Temizleme ve Özellik Türetme

## Bu notebook ne yapıyor?

NB1'de veriyi *tanıdık*. Şimdi onu *kullanışlı* hale getireceğiz.

Ham verinin sorunları:
1. **27 sütun tamamen boş** → silinecek (162 → 135 sütun)
2. **Tarihler sayı formatında** (`1012023` gibi) → gerçek tarihe çevrilecek
3. **"Hasta kaç gün yattı?", "Toplam fatura ne kadar?"** → henüz yok → türetilecek

**Bu notebook'un çıktısı:** `data/processed/hcp_clean.parquet`  
NB3 (görsel analiz) ve NB4 (makine öğrenmesi modeli) bu dosyayı kullanacak.


---
## 2.0 — Veri Yükleme ve Boş Sütun Temizliği

**Analoji:** Masanızda 162 klasör var. NB1 size "27 tanesi tamamen boş" dedi.  
İlk iş: o boş klasörleri çöpe atmak. Masa temizlenir, çalışmak kolaylaşır.

NB1'in ürettiği `null_summary.csv`'yi okuyup, `null_pct == 100` olan sütunları drop ediyoruz.  
Bu sayede hangi sütunları neden sildiğimiz kayıt altında — kararımız **belgelenmiş**.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Ham veriyi yükle
df = pd.read_parquet(ROOT / "data/processed/hcp.parquet")
print(f"Ham veri: {df.shape[0]:,} satır × {df.shape[1]} sütun")


Ham veri: 30,615 satır × 162 sütun


In [2]:
# NB1'in ürettiği null_summary.csv'den %100 boş sütunları al
null_summary = pd.read_csv(ROOT / "reports/null_summary.csv")
drop_cols = null_summary.loc[null_summary["null_pct"] >= 99.9, "column"].tolist()

df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(f"Çıkarılan sütun sayısı : {len(drop_cols)}")
print(f"Kalan                  : {df.shape[0]:,} satır × {df.shape[1]} sütun")
print(f"\nÇıkarılan sütunlar (tümü):")
for i, c in enumerate(drop_cols, 1):
    print(f"  {i:2d}. {c}")


Çıkarılan sütun sayısı : 64
Kalan                  : 30,615 satır × 98 sütun

Çıkarılan sütunlar (tümü):
   1. AdditionalDiagnosis49
   2. Procedure33
   3. Procedure35
   4. Procedure36
   5. Procedure37
   6. Procedure38
   7. Procedure39
   8. Procedure40
   9. Procedure41
  10. Procedure42
  11. Procedure43
  12. Procedure44
  13. Procedure45
  14. Procedure46
  15. Procedure47
  16. Procedure48
  17. Procedure49
  18. AdditionalDiagnosis42
  19. Procedure50
  20. AdditionalDiagnosis43
  21. DRG_Version
  22. AdditionalDiagnosis44
  23. AdditionalDiagnosis45
  24. AdditionalDiagnosis46
  25. AdditionalDiagnosis47
  26. AdditionalDiagnosis48
  27. Procedure34
  28. Procedure32
  29. Procedure31
  30. Procedure26
  31. Procedure30
  32. Procedure27
  33. Procedure28
  34. Procedure29
  35. AdditionalDiagnosis40
  36. AdditionalDiagnosis41
  37. AdditionalDiagnosis39
  38. Procedure25
  39. Procedure24
  40. AdditionalDiagnosis34
  41. AdditionalDiagnosis35
  42. AdditionalDiagnosis36

---
## 2.1 — Tarihleri Parse Et (DDMMYYYY → datetime)

**Problem:** `AdmissionDate` sütununda `1012023` gibi sayılar var.  
Bu aslında **01/01/2023** (1 Ocak 2023) demek — ama Python bunu bilmiyor.

**HCP format kuralı (DDMMYYYY):**
```
1012023  →  DD=01  MM=01  YYYY=2023  →  2023-01-01
31122023 →  DD=31  MM=12  YYYY=2023  →  2023-12-31
```

Formülü elle yapalım:
```
gün  = n ÷ 1,000,000          (tam bölme)
ay   = (n ÷ 10,000) mod 100
yıl  = n mod 10,000
```

**Neden bu dönüşüm önemli?**  
Sayılarla `1012023 - 6012023` diyemezsiniz; bu anlamsız.  
Ama datetime ile `SeparationDate - AdmissionDate` = kaç gün yattı sorusu tek satırda cevaplanır.

**Hata bayrağı:** Çıkış tarihi giriş tarihinden önce gelen satırlar → `date_error = True`  
Bu satırları silmiyoruz; modelde gürültü yaratır ama şeffaflık için bayrağı koyuyoruz.


In [3]:
def parse_hcp_date(series: pd.Series) -> pd.Series:
    """
    DDMMYYYY formatındaki integer sütununu datetime'a çevirir.

    Örnek:
        1012023  → gün=1,  ay=01, yıl=2023 → 2023-01-01
        31122023 → gün=31, ay=12, yıl=2023 → 2023-12-31
        12061955 → gün=12, ay=06, yıl=1955 → 1955-06-12
    """
    s = pd.to_numeric(series, errors="coerce")

    day  = (s // 1_000_000).astype("Int64")
    mon  = (s // 10_000 % 100).astype("Int64")
    yr   = (s % 10_000).astype("Int64")

    # "2023-01-01" formatına dönüştür, pandas bunu parse eder
    date_str = (
        yr.astype(str).str.zfill(4) + "-"
        + mon.astype(str).str.zfill(2) + "-"
        + day.astype(str).str.zfill(2)
    )
    return pd.to_datetime(date_str, format="%Y-%m-%d", errors="coerce")


df["AdmissionDate_dt"]  = parse_hcp_date(df["AdmissionDate"])
df["SeparationDate_dt"] = parse_hcp_date(df["SeparationDate"])
df["DateOfBirth_dt"]    = parse_hcp_date(df["DateOfBirth"])

# Tarih hatası bayrağı: çıkış tarihi < giriş tarihi
df["date_error"] = df["SeparationDate_dt"] < df["AdmissionDate_dt"]

print("Tarih parse sonuçları:")
for col in ["AdmissionDate_dt", "SeparationDate_dt", "DateOfBirth_dt"]:
    valid = df[col].notna().sum()
    print(f"  {col:<22}: {valid:,} geçerli / {len(df):,} toplam"
          f"  ({valid/len(df)*100:.1f}%)")
print(f"\n  date_error (sep < adm): {df['date_error'].sum():,} satır"
      f"  ({df['date_error'].mean()*100:.2f}%)")

# Görsel kontrol: ilk 4 satır
print("\nÖrnek dönüşümler:")
sample = df[["AdmissionDate","AdmissionDate_dt",
             "SeparationDate","SeparationDate_dt"]].head(4)
print(sample.to_string(index=False))


Tarih parse sonuçları:
  AdmissionDate_dt      : 30,615 geçerli / 30,615 toplam  (100.0%)
  SeparationDate_dt     : 30,615 geçerli / 30,615 toplam  (100.0%)
  DateOfBirth_dt        : 30,615 geçerli / 30,615 toplam  (100.0%)

  date_error (sep < adm): 0 satır  (0.00%)

Örnek dönüşümler:
 AdmissionDate AdmissionDate_dt  SeparationDate SeparationDate_dt
       1012023       2023-01-01         2012023        2023-01-02
       1012023       2023-01-01         6012023        2023-01-06
       1012023       2023-01-01         9012023        2023-01-09
       1022023       2023-02-01         1022023        2023-02-01


---
## 2.2 — LOS ve Age Hesapla

### LOS (Length of Stay — Yatış Süresi)

```
LOS = SeparationDate − AdmissionDate  (gün cinsinden)
```

- **LOS = 0** → günübirlik hasta (same-day) — hata **değil**, normaldir!  
  Bu hastanenin %78'i günübirlik, yani LOS=0 en yaygın değer.
- **LOS > 30** → uzun yatış; palyatif bakım / komplikasyon olabilir.

### Age (Yatış Anındaki Yaş)

```
Age = AdmissionDate.yıl − DateOfBirth.yıl
```

Ama dikkat: doğum günü o yıl henüz geçmediyse yaş 1 eksiğidir.  
→ Ay ve günü de kontrol ederek **tam yaş** düzeltmesi yapıyoruz.

Örnek: Doğum 15 Haziran 1980, Yatış 10 Mart 2023  
→ Ham hesap: 2023 - 1980 = 43  
→ Haziran henüz gelmedi → gerçek yaş = **42**


In [4]:
# LOS (gün)
df["LOS"] = (df["SeparationDate_dt"] - df["AdmissionDate_dt"]).dt.days

# Özet istatistikler
los_neg    = (df["LOS"] < 0).sum()
los_zero   = (df["LOS"] == 0).sum()
los_1_7    = ((df["LOS"] >= 1)  & (df["LOS"] <= 7)).sum()
los_8_30   = ((df["LOS"] >= 8)  & (df["LOS"] <= 30)).sum()
los_gt30   = (df["LOS"] > 30).sum()

print("LOS (Yatış Günü) Dağılımı:")
print(f"  Negatif  (<0, veri hatası)  : {los_neg:,}  ({los_neg/len(df)*100:.1f}%)")
print(f"  Günübirlik (=0)             : {los_zero:,}  ({los_zero/len(df)*100:.1f}%)")
print(f"  Kısa yatış (1–7 gün)        : {los_1_7:,}  ({los_1_7/len(df)*100:.1f}%)")
print(f"  Orta yatış (8–30 gün)       : {los_8_30:,}  ({los_8_30/len(df)*100:.1f}%)")
print(f"  Uzun yatış (>30 gün)        : {los_gt30:,}  ({los_gt30/len(df)*100:.1f}%)")
print(f"\n  Ortalama LOS : {df['LOS'].mean():.2f} gün")
print(f"  Medyan LOS   : {df['LOS'].median():.0f} gün")
print(f"  Max LOS      : {df['LOS'].max():.0f} gün")


LOS (Yatış Günü) Dağılımı:
  Negatif  (<0, veri hatası)  : 0  (0.0%)
  Günübirlik (=0)             : 24,017  (78.4%)
  Kısa yatış (1–7 gün)        : 6,107  (19.9%)
  Orta yatış (8–30 gün)       : 453  (1.5%)
  Uzun yatış (>30 gün)        : 38  (0.1%)

  Ortalama LOS : 0.71 gün
  Medyan LOS   : 0 gün
  Max LOS      : 99 gün


In [5]:
# Age (yatış anındaki yaş)
adm_y, adm_m, adm_d = (df["AdmissionDate_dt"].dt.year,
                        df["AdmissionDate_dt"].dt.month,
                        df["AdmissionDate_dt"].dt.day)
dob_y, dob_m, dob_d = (df["DateOfBirth_dt"].dt.year,
                        df["DateOfBirth_dt"].dt.month,
                        df["DateOfBirth_dt"].dt.day)

df["Age"] = adm_y - dob_y  # ham yaş

# Doğum günü o yıl henüz geçmediyse 1 çıkar
birthday_passed = (adm_m > dob_m) | ((adm_m == dob_m) & (adm_d >= dob_d))
df["Age"] = df["Age"].where(birthday_passed, df["Age"] - 1)

# Özet
age_groups = {
    "Yenidoğan (0)"       : (df["Age"] == 0).sum(),
    "Çocuk (1–17)"        : ((df["Age"] >= 1) & (df["Age"] <= 17)).sum(),
    "Genç yetişkin (18–44)": ((df["Age"] >= 18) & (df["Age"] <= 44)).sum(),
    "Orta yaş (45–64)"    : ((df["Age"] >= 45) & (df["Age"] <= 64)).sum(),
    "Yaşlı (65+)"         : (df["Age"] >= 65).sum(),
    "Hata (<0 veya >120)" : ((df["Age"] < 0) | (df["Age"] > 120)).sum(),
}
print("Age (Yatış Anındaki Yaş) Dağılımı:")
for label, n in age_groups.items():
    print(f"  {label:<28}: {n:,}  ({n/len(df)*100:.1f}%)")

print(f"\n  Ortalama yaş : {df['Age'].mean():.1f}")
print(f"  Medyan yaş   : {df['Age'].median():.0f}")
print(f"  Min / Max    : {df['Age'].min()} / {df['Age'].max()}")


Age (Yatış Anındaki Yaş) Dağılımı:
  Yenidoğan (0)               : 340  (1.1%)
  Çocuk (1–17)                : 693  (2.3%)
  Genç yetişkin (18–44)       : 3,693  (12.1%)
  Orta yaş (45–64)            : 9,098  (29.7%)
  Yaşlı (65+)                 : 16,791  (54.8%)
  Hata (<0 veya >120)         : 0  (0.0%)

  Ortalama yaş : 62.5
  Medyan yaş   : 67
  Min / Max    : 0 / 105


---
## 2.3 — Komorbidite ve Prosedür Sayısı

### Komorbidite nedir?

Hastanın ana hastalığına ek, **eşlik eden diğer hastalıkları** komorbidite denir.

Örnek: Kalp ameliyatı için yatan bir hasta aynı zamanda:
- Diyabetliyse → 1 komorbidite  
- Hem diyabetli hem hipertansifse → 2 komorbidite

HCP'de `AdditionalDiagnosis1` ... `AdditionalDiagnosis41` sütunları var.  
Her dolu sütun = 1 ek tanı.

**Neden önemli?**  
Komorbidite sayısı ↑ → Bakım karmaşıklığı ↑ → Maliyet ↑  
Bu, modelimizin en güçlü tahmin özelliklerinden biri olacak.

### Prosedür sayısı

`Procedure1` ... `Procedure32` sütunları.  
Her dolu sütun = hastaya uygulanan 1 cerrahi/tıbbi işlem.


In [6]:
# Boşluk tespiti (NB1'deki aynı mantık)
NULL_LIKE = {"", "nan", "none", "na", "<na>", "nat", "null"}

def is_filled(series: pd.Series) -> "pd.Series[bool]":
    """True = sütun gerçekten dolu, False = boş/whitespace/NaN"""
    s = series.astype(str).str.strip().str.lower()
    return ~s.isin(NULL_LIKE)


# Kalan tanı ve prosedür sütunlarını bul
diag_cols = sorted([c for c in df.columns if c.startswith("AdditionalDiagnosis")])
proc_cols  = sorted([c for c in df.columns if c.startswith("Procedure")])

print(f"Tanı sütunları    : {len(diag_cols)} adet  "
      f"({diag_cols[0]} → {diag_cols[-1]})")
print(f"Prosedür sütunları: {len(proc_cols)} adet  "
      f"({proc_cols[0]} → {proc_cols[-1]})")


Tanı sütunları    : 22 adet  (AdditionalDiagnosis1 → AdditionalDiagnosis9)
Prosedür sütunları: 14 adet  (Procedure1 → Procedure9)


In [7]:
# Sayım: her satır için kaç tanesi dolu?
# np.stack → (n_rows, n_cols) boolean matris → .sum(axis=1) = satır toplamı
diag_matrix = np.stack(
    [is_filled(df[c]).to_numpy(dtype=bool, na_value=False) for c in diag_cols],
    axis=1
)
proc_matrix = np.stack(
    [is_filled(df[c]).to_numpy(dtype=bool, na_value=False) for c in proc_cols],
    axis=1
)

df["comorbidity_count"] = diag_matrix.sum(axis=1)
df["procedure_count"]   = proc_matrix.sum(axis=1)

print("Komorbidite Sayısı (değer → kaç hasta):")
print(df["comorbidity_count"].value_counts().sort_index().head(12).to_string())

print("\nProsedür Sayısı (değer → kaç hasta):")
print(df["procedure_count"].value_counts().sort_index().head(12).to_string())

print(f"\nKomorbidite — Ort: {df['comorbidity_count'].mean():.2f}"
      f"  Medyan: {df['comorbidity_count'].median():.0f}"
      f"  Max: {df['comorbidity_count'].max()}")
print(f"Prosedür      — Ort: {df['procedure_count'].mean():.2f}"
      f"  Medyan: {df['procedure_count'].median():.0f}"
      f"  Max: {df['procedure_count'].max()}")


Komorbidite Sayısı (değer → kaç hasta):
comorbidity_count
0     11173
1      5356
2      5375
3      3364
4      2133
5      1221
6       645
7       412
8       254
9       165
10      124
11       90

Prosedür Sayısı (değer → kaç hasta):
procedure_count
0       821
1     16531
2      4962
3      3319
4      2239
5      1066
6       661
7       374
8       186
9        66
10      131
11      170



Komorbidite — Ort: 1.90  Medyan: 1  Max: 22
Prosedür      — Ort: 2.08  Medyan: 1  Max: 14


---
## 2.4 — Toplam Ücret ($AUD)

### HCP'de ücretler nasıl saklanıyor?

**Kuruş (cent) cinsinden tam sayı** olarak.

```
AccommodationCharge = 35710  →  $357.10 AUD
PharmacyCharge      =  4250  →   $42.50 AUD
```

Neden kuruş? Çünkü `35710` (integer) `357.10` (float)'dan daha az yer tutar ve  
kayan nokta hatası olmaz. Bu hastane veri tabanlarında yaygın bir yaklaşımdır.

**Dönüşüm:** `÷ 100`

**Hedef değişken:** `total_charge_aud` — tüm charge sütunlarının toplamının AUD cinsinden karşılığı.  
NB4'teki modelimiz bu değeri **tahmin etmeye** çalışacak.


In [8]:
# 'Charge' içeren sütunları bul
charge_cols = [c for c in df.columns if "Charge" in c]
print(f"Ücret sütunları ({len(charge_cols)} adet):")
print("-" * 55)
for c in charge_cols:
    median_cents = df[c].median()
    print(f"  {c:<28}: medyan {median_cents:>8,.0f} kuruş"
          f" = ${median_cents/100:>8,.2f} AUD")


Ücret sütunları (11 adet):
-------------------------------------------------------
  AccommodationCharge         : medyan   35,709 kuruş = $  357.09 AUD
  TheatreCharge               : medyan        0 kuruş = $    0.00 AUD
  LabourWardCharge            : medyan        0 kuruş = $    0.00 AUD
  ICU_Charge                  : medyan        0 kuruş = $    0.00 AUD
  ProsthesisCharge            : medyan        0 kuruş = $    0.00 AUD
  PharmacyCharge              : medyan        0 kuruş = $    0.00 AUD
  OtherCharges                : medyan        0 kuruş = $    0.00 AUD
  BundledCharges              : medyan        0 kuruş = $    0.00 AUD
  HIH_Charges                 : medyan        0 kuruş = $    0.00 AUD
  SCN_Charges                 : medyan        0 kuruş = $    0.00 AUD
  CCU_Charges                 : medyan        0 kuruş = $    0.00 AUD


In [9]:
# Toplam ücret: tüm charge sütunlarını topla, kuruş → AUD
df["total_charge_aud"] = df[charge_cols].sum(axis=1) / 100

tc = df["total_charge_aud"]
print("total_charge_aud İstatistikleri:")
print(f"  Ortalama : ${tc.mean():>10,.2f}")
print(f"  Medyan   : ${tc.median():>10,.2f}")
print(f"  Std      : ${tc.std():>10,.2f}")
print(f"  Min      : ${tc.min():>10,.2f}")
print(f"  %25      : ${tc.quantile(0.25):>10,.2f}")
print(f"  %75      : ${tc.quantile(0.75):>10,.2f}")
print(f"  %95      : ${tc.quantile(0.95):>10,.2f}")
print(f"  Max      : ${tc.max():>10,.2f}")
print(f"\n  Sıfır ücretli epizod: {(tc == 0).sum():,}"
      f"  ({(tc==0).mean()*100:.1f}%)")


total_charge_aud İstatistikleri:
  Ortalama : $  2,685.46
  Medyan   : $    650.00
  Std      : $  4,389.14
  Min      : $      0.00
  %25      : $    357.10
  %75      : $  2,942.00
  %95      : $ 12,559.90
  Max      : $ 69,138.00

  Sıfır ücretli epizod: 430  (1.4%)


---
## 2.5 — MDC (Ana Tanı Kategorisi)

### DRG nedir?

DRG = Diagnosis-Related Group (Tanıya İlişkin Grup).  
Her epizoda, hangi kaynakları tükettiğini özetleyen bir DRG kodu atanır.

```
D12B  →  MDC = D  =  Kulak, Burun, Boğaz
F06A  →  MDC = F  =  Dolaşım Sistemi
I03B  →  MDC = I  =  Kas-İskelet Sistemi
```

**Neden MDC'ye indirgiyoruz?**  
DRG'de 500+ farklı kod var; modelde çok fazla kategori analizi zorlaştırır.  
MDC ile bunları **26 ana gruba** düşürüyoruz.


In [10]:
# AR-DRG v10 MDC eşleştirme tablosu (Australian Refined DRG)
MDC_MAP = {
    "A": "Pre-MDC (Nakil / Trakeostomi / ECMO)",
    "B": "Sinir Sistemi Hastalıkları",
    "C": "Göz Hastalıkları",
    "D": "Kulak, Burun, Boğaz ve Ağız",
    "E": "Solunum Sistemi Hastalıkları",
    "F": "Dolaşım Sistemi Hastalıkları",
    "G": "Sindirim Sistemi Hastalıkları",
    "H": "Karaciğer, Safra Kesesi ve Pankreas",
    "I": "Kas-İskelet Sistemi Hastalıkları",
    "J": "Deri ve Deri Altı Doku",
    "K": "Endokrin, Beslenme ve Metabolizma",
    "L": "Böbrek ve Üriner Sistem",
    "M": "Erkek Üreme Sistemi",
    "N": "Kadın Üreme Sistemi",
    "O": "Gebelik, Doğum ve Lohusalık",
    "P": "Yenidoğan",
    "Q": "Kan ve Kan Yapıcı Organ Hastalıkları",
    "R": "Ruhsal Hastalıklar",
    "S": "Madde Kullanımı ve Bağımlılık",
    "T": "Enfeksiyöz ve Parazitik Hastalıklar",
    "U": "Yanıklar",
    "V": "Sağlık Durumunu Etkileyen Faktörler",
    "W": "Yaralanma, Zehirlenme ve Toksik Etkiler",
    "X": "Diğer Faktörler",
    "Y": "HIV Enfeksiyonları",
    "Z": "Gruplandırılamayan",
}

df["DRG"]       = df["DRG"].astype(str).str.strip()
df["MDC"]       = df["DRG"].str[0].str.upper()
df["MDC_label"] = df["MDC"].map(MDC_MAP).fillna("Bilinmiyor")

mdc_dist = (df.groupby(["MDC", "MDC_label"])
              .size()
              .reset_index(name="count")
              .assign(pct=lambda d: d["count"] / len(df) * 100)
              .sort_values("count", ascending=False))

print(f"MDC Dağılımı (toplam {df['MDC'].nunique()} kategori, en sık 12):")
print("-" * 72)
print(f"{'MDC':<4}  {'Kategori':<40}  {'Sayı':>6}  {'%':>5}")
print("-" * 72)
for _, row in mdc_dist.head(12).iterrows():
    print(f"  {row['MDC']:<3}  {row['MDC_label']:<40}  "
          f"{row['count']:>6,}  {row['pct']:>5.1f}%")


MDC Dağılımı (toplam 23 kategori, en sık 12):
------------------------------------------------------------------------
MDC   Kategori                                    Sayı      %
------------------------------------------------------------------------
  L    Böbrek ve Üriner Sistem                   10,001   32.7%
  R    Ruhsal Hastalıklar                         6,965   22.8%
  I    Kas-İskelet Sistemi Hastalıkları           2,945    9.6%
  G    Sindirim Sistemi Hastalıkları              1,707    5.6%
  F    Dolaşım Sistemi Hastalıkları               1,691    5.5%
  C    Göz Hastalıkları                           1,549    5.1%
  D    Kulak, Burun, Boğaz ve Ağız                1,004    3.3%
  N    Kadın Üreme Sistemi                          662    2.2%
  Z    Gruplandırılamayan                           582    1.9%
  J    Deri ve Deri Altı Doku                       551    1.8%
  O    Gebelik, Doğum ve Lohusalık                  513    1.7%
  E    Solunum Sistemi Hastalıkları       

---
## 2.6 — Temiz Veriyi Kaydet

Tüm dönüşümleri tamamladık. Şimdi hcp_clean.parquet'i kaydediyoruz.

**Eklenen yeni sütunlar (11 adet):**

| Sütun | Tip | Açıklama |
|---|---|---|
| `AdmissionDate_dt` | datetime | Giriş tarihi |
| `SeparationDate_dt` | datetime | Çıkış tarihi |
| `DateOfBirth_dt` | datetime | Doğum tarihi |
| `date_error` | bool | `sep < adm` ise True |
| `LOS` | int | Yatış süresi (gün) |
| `Age` | int | Yatış anındaki yaş |
| `comorbidity_count` | int | Ek tanı sayısı (0–41) |
| `procedure_count` | int | Prosedür sayısı (0–32) |
| `total_charge_aud` | float | Toplam fatura ($AUD) |
| `MDC` | str | Ana tanı kategorisi harfi |
| `MDC_label` | str | MDC Türkçe açıklaması |


In [11]:
out_path = ROOT / "data/processed/hcp_clean.parquet"
df.to_parquet(out_path, index=False)

import os
size_mb = os.path.getsize(out_path) / 1024**2
print(f"✓ Kaydedildi : {out_path}")
print(f"  Boyut      : {size_mb:.2f} MB")
print(f"  Şekil      : {df.shape[0]:,} satır × {df.shape[1]} sütun")

new_cols = [
    "AdmissionDate_dt","SeparationDate_dt","DateOfBirth_dt",
    "date_error","LOS","Age","comorbidity_count","procedure_count",
    "total_charge_aud","MDC","MDC_label"
]
print(f"  Yeni sütun : {len(new_cols)} adet → {new_cols}")


✓ Kaydedildi : /Users/osmanorka/SJGHC-Case-Study/data/processed/hcp_clean.parquet
  Boyut      : 1.18 MB
  Şekil      : 30,615 satır × 109 sütun
  Yeni sütun : 11 adet → ['AdmissionDate_dt', 'SeparationDate_dt', 'DateOfBirth_dt', 'date_error', 'LOS', 'Age', 'comorbidity_count', 'procedure_count', 'total_charge_aud', 'MDC', 'MDC_label']


In [12]:
# Nihai özet — model için önemli sütunların kontrol tablosu
print("=" * 60)
print("NB2 TAMAMLANDI — MODEL ÖNCESİ KONTROL")
print("=" * 60)

checks = [
    ("Hedef (y)",         f"total_charge_aud — Ort: ${df['total_charge_aud'].mean():,.0f}"),
    ("LOS",               f"Ort: {df['LOS'].mean():.1f} gün  Medyan: {df['LOS'].median():.0f}"),
    ("Age",               f"Ort: {df['Age'].mean():.1f}  Medyan: {df['Age'].median():.0f}"),
    ("Komorbidite",       f"Ort: {df['comorbidity_count'].mean():.2f}  Max: {df['comorbidity_count'].max()}"),
    ("Prosedür",          f"Ort: {df['procedure_count'].mean():.2f}  Max: {df['procedure_count'].max()}"),
    ("MDC kategorisi",    f"{df['MDC'].nunique()} farklı  En sık: {df['MDC'].value_counts().idxmax()}"),
    ("Tarih hatası",      f"{df['date_error'].sum():,} satır  ({df['date_error'].mean()*100:.2f}%)"),
    ("Toplam satır",      f"{len(df):,}"),
    ("Toplam sütun",      f"{df.shape[1]}"),
]
for k, v in checks:
    print(f"  {k:<22}: {v}")
print("=" * 60)
print("\n→ Sıradaki: NB3 (Keşifsel Veri Analizi + Sunum Grafikleri)")


NB2 TAMAMLANDI — MODEL ÖNCESİ KONTROL
  Hedef (y)             : total_charge_aud — Ort: $2,685
  LOS                   : Ort: 0.7 gün  Medyan: 0
  Age                   : Ort: 62.5  Medyan: 67
  Komorbidite           : Ort: 1.90  Max: 22
  Prosedür              : Ort: 2.08  Max: 14
  MDC kategorisi        : 23 farklı  En sık: L
  Tarih hatası          : 0 satır  (0.00%)
  Toplam satır          : 30,615
  Toplam sütun          : 109

→ Sıradaki: NB3 (Keşifsel Veri Analizi + Sunum Grafikleri)
